<div class="alert alert-block alert-info" style="font-size:20px">
    <b>We will be writing our own R code</b><br>
    <ul>
        <li><b>Open</b> the <i>3. CLEAN_JOIN_diabetes_ptsd_cohorts.ipynb</i> file in <b>GitHub</b> where you can view then copy/paste the code chunks into a new Jupyter Notebook
            <ul>
</div>

# Notebook File Overview

In this notebook, we will be importing the data frames we created in the first two notebooks:
1. person_control.csv
2. person_study.csv
3. survey_control.csv
4. survey_study.csv

We will use the code snippet provided by *All of Us* to import our data frames from the workspace bucket (looks similar to the snippet used to save our data frames to the workspace bucket). 

We will do some data processing and cleaning on our existing data frames including:
* Sort both cohorts byt person_ID
* Clean up answer and question columns
* remove duplicate person_IDs
* Create a new variable to designate which participants **have** and **do not have** diabetes 
* Combine the two dataframes into one dataframe
* Calculate participant age based on the current date and their DOB and create a new variable called *age*
* Select the final columns we want for our final dataframe we will analyze

# Add the code snippet from the *All of Us R and Cloud Storage snippets*

**Step 1: Run *Setup***

In [ ]:
library(tidyverse)  # Data wrangling packages.

**Step 2: Run the *copy_file_from_workspace_bucket.R* code snippet**

This will import:

* person_control.csv
* person_study.csv
* survey_control.csv
* survey_study.csv

as data frames

**NOTE: You will have to add another line of code to each step to get both files imported**

<div style=" background-color:#f8d7da; color:#842029; border:1px solid #f5c2c7; border-radius:8px; padding:20px; font-size:20px; "> 
<b>We are going to update the provided snippet, even past the <i>DON'T CHANGE FROM HERE</i> Warning</b><br> 
</div>

In [ ]:
# This snippet assumes that you run setup first

# This code copies a file from your Google Bucket into a dataframe

# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)

## CHANGE ORIGINAL TO THIS 
## ORIGINAL: name_of_file_in_bucket <- 'test.csv'
person_control <- 'person_control.csv'
person_study <- 'person_study.csv'
survey_control <- 'survey_control.csv'
survey_study <- 'survey_study.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket

## CHANGE ORIGINAL TO THIS 
## ORIGINAL: system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", person_control, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", person_study, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", survey_control, " ."), intern=T)
system(paste0("gsutil cp ", my_bucket, "/data/", survey_study, " ."), intern=T)

# Load the file into a dataframe

## CHANGE ORIGINAL TO THIS 
## ORIGINAL: my_dataframe  <- read_csv(name_of_file_in_bucket)
person_control  <- read_csv(person_control)
person_study  <- read_csv(person_study)
survey_control  <- read_csv(survey_control)
survey_study  <- read_csv(survey_study)


## CHANGE ORIGINAL TO THIS 
## ORIGINAL: head(my_dataframe)
head(person_control)
head(person_study)
head(survey_control)
head(survey_study)

# Let's clean and join our data before saving to our workspace bucket

**Step 1: Sort our survey_control and survey_study data by *person_id***

Create a new object called survey_arranged using the assignment (<-) operator

Then, we will use the *arrange()* function from the **Dplyr** package within the **Tidyverse** (loaded earlier)

In [ ]:
survey_control_arranged <- survey_control %>%
arrange(person_id)

survey_study_arranged <- survey_study %>%
arrange(person_id)

In [ ]:
dim(survey_control_arranged)
head(survey_control_arranged, 5)

dim(survey_study_arranged)
head(survey_study_arranged, 5)

**Step 2: Clean up the answer columns**

We will create new objects called survey_control_clean and survey_study_clean which are modified dataframes from the previous survey_control_arranged and survey_study_arranged dataframes

As you can see in the previous chunk, the answer column includes the same string of text found in the question column which we want to **remove**

We will use the *str_remove()* and *str_trim()* functions from the **stringr** package within the **Tidyverse**

In [ ]:
survey_control_clean <- survey_control_arranged %>%
  mutate(
    answer = str_remove(answer, fixed(question)), # remove the exact question text (literal match)
    answer = str_remove(answer, "^\\s*-\\s*"),    # remove a leading hyphen and surrounding spaces left behind
    answer = str_trim(answer)                     # trim any leftover whitespace
  )


survey_study_clean <- survey_study_arranged %>%
  mutate(
    answer = str_remove(answer, fixed(question)), # remove the exact question text (literal match)
    answer = str_remove(answer, "^\\s*-\\s*"),    # remove a leading hyphen and surrounding spaces left behind
    answer = str_trim(answer)                     # trim any leftover whitespace
  )

In [ ]:
head(survey_control_clean)
head(survey_study_clean)

**Step 3: Make the values in the question column the column names and the values in the answer column the values for the new columns**

This will get rid of the person_id duplicates but keep both the questions and their responses

We will create a new object called survey_wide which is modifies the previous survey_clean data frame

We will use the *pivot_wider()* function from the **tidyr** package within the **Tidyverse** to accomplish this

In [ ]:
survey_control_wide <- survey_control_clean %>%
  pivot_wider(
    names_from = question,
    values_from = answer
  )

survey_study_wide <- survey_study_clean %>%
  pivot_wider(
    names_from = question,
    values_from = answer
  )

In [ ]:
dim(survey_control_wide)
head(survey_control_wide, 5)

dim(survey_study_wide)
head(survey_study_wide, 5)

**Step 4: Join the updated survey dataframes (survey_control_wide and survey_study_wide), to their respective demographic variable dataframe (person_control and person_study)**

We will create two new objects called *diabetes_control_ptsd* and *diabetes_control_ptsd* which will include variables from our survey and demographic variable dataframes.

We will use the *left_join()* function from the **dplyr** package to join the two data frames on the **person_id** variable.

We will also use the *filter()* function from the **dplyr** package to remove NA values from our new data frame.

In [ ]:
library(dplyr)

diabetes_control_ptsd <- person_control %>%
  left_join(survey_control_wide, by = c("person_id" = "person_id")) %>%
  filter(
    !is.na(`Are you still seeing a doctor or health care provider for post-traumatic stress disorder (PTSD)?`),
    !is.na(`Are you currently prescribed medications and/or receiving treatment for post-traumatic stress disorder (PTSD)?`)
  )

diabetes_study_ptsd <- person_study %>%
  left_join(survey_study_wide, by = c("person_id" = "person_id")) %>%
  filter(
    !is.na(`Are you still seeing a doctor or health care provider for post-traumatic stress disorder (PTSD)?`),
    !is.na(`Are you currently prescribed medications and/or receiving treatment for post-traumatic stress disorder (PTSD)?`)
  )

In [ ]:
dim(diabetes_control_ptsd)
head(diabetes_control_ptsd, 5)

dim(diabetes_study_ptsd)
head(diabetes_study_ptsd, 5)

# Clean and join the two dataframes

**Step 1: Add new columns to each data frame called *diabetes* and include values based on diabetes status**

We will use the mutate() function from the **dplyr** package to create new variables for each data frame.

In [ ]:
library(dplyr)

diabetes_no_ptsd <- diabetes_control_ptsd %>%
  mutate(diabetes = "No")

diabetes_yes_ptsd <- diabetes_study_ptsd%>%
  mutate(diabetes = "Yes")

dim(diabetes_no_ptsd)
head(diabetes_no_ptsd)
dim(diabetes_yes_ptsd)
head(diabetes_yes_ptsd)

**Step 2: Combine data frames and finalize dataset**

We will use several functions from the **dplyr** package to create our final analysis-ready dataset:

* Use the *bind_rows()* function to combine the diabetes_no_ptsd and diabetes_yes_ptsd data frames into one dataset
* Use the *rename()* function to give shorter, more manageable names to the long PTSD survey question columns
* Use the *mutate()* function to calculate participant age from their date of birth and create a new variable called *age*
* Use the *select()* function to reorder columns and remove the original date_of_birth column since we now have the calculated age


We use bind_rows() instead of a join function because the column names for each data frame are the same and in the same order, so we can simply stack the data together which is easier, cleaner, and has fewer potential errors

In [ ]:
final_diabetes_ptsd <- bind_rows(diabetes_no_ptsd, diabetes_yes_ptsd) %>%
  rename(
    ptsd_doctor = `Are you still seeing a doctor or health care provider for post-traumatic stress disorder (PTSD)?`,
    ptsd_treatment = `Are you currently prescribed medications and/or receiving treatment for post-traumatic stress disorder (PTSD)?`
  ) %>%
  mutate(
    age = floor(interval(ymd_hms(date_of_birth), today()) / years(1))
  ) %>%
select(person_id, age, everything(), -date_of_birth)



dim(final_diabetes_ptsd)
head(final_diabetes_ptsd)

# Add the code snippet from the *All of Us R and Cloud Storage snippets*

<div class="alert alert-block alert-info" style="font-size:16px">
    
<b>Navigate to the code snippets</b>
    <ul style="margin-top: 0px;">    
        <li>All of Us R and Cloud Storage snippets -> Copy file to or from Workspace Bucket -> copy_data_to_workspace_bucket.R</li>
    </ul>
    
<b>Read the code carefully</b>
    <ul style="margin-top: 0px;">    
        <li>Replace <b>df</b> with <i>THE NAME OF YOUR DATAFRAME</i>
            <ul>
                <li><i>final_diabetes_ptsd</i></li>
            </ul>
        </li>
        <li>Replace <b>'test.csv'</b> with <i>THE NAME of the file</i> you're going to store in the bucket (don't delete the quotation marks)
            <ul>
                <li><i>'final_diabetes_ptsd.csv'</i></li>
            </ul>
        </li>
    </ul>
        
</div>

In [ ]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- final_diabetes_ptsd

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- 'final_diabetes_ptsd.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)
